In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn.functional as F



In [32]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5,0.5,0.5),
        (0.5,0.5,0.5)
    )
])


In [33]:
train_dataset = datasets.ImageFolder("../Data/Images/Training", transform=transform)
val_dataset = datasets.ImageFolder("../Data/Images/Validation", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class_names = train_dataset.classes
print(class_names)


['Adult Naruto', 'Teen Naruto', 'Young Naruto']


In [36]:

class SimpleCNN(nn.Module):
    def __init__(self, num_classes, img_size=128):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # Fully connected layers
        self.fc1 = nn.Linear(128*8*8, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84,10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.flatten(x,1)  # flatten
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.log_softmax(self.fc3(x),dim=1)
        return x
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN(num_classes=len(class_names))
model.to(device)


SimpleCNN(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=8192, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [37]:
from torchsummary import summary
summary(model, input_size=(3, 32, 32))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 64, 32, 32]           1,792
         MaxPool2d-2           [-1, 64, 16, 16]               0
            Conv2d-3          [-1, 128, 16, 16]          73,856
         MaxPool2d-4            [-1, 128, 8, 8]               0
            Linear-5                  [-1, 120]         983,160
            Linear-6                   [-1, 84]          10,164
            Linear-7                   [-1, 10]             850
Total params: 1,069,822
Trainable params: 1,069,822
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.01
Forward/backward pass size (MB): 0.94
Params size (MB): 4.08
Estimated Total Size (MB): 5.03
----------------------------------------------------------------


In [39]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [41]:
epochs = 10
print("Training")
for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100 * correct / total
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}, Accuracy: {train_acc:.2f}%")


Training
Epoch 1, Loss: 0.5698, Accuracy: 76.42%
Epoch 2, Loss: 0.4332, Accuracy: 82.98%
Epoch 3, Loss: 0.3032, Accuracy: 88.25%
Epoch 4, Loss: 0.2039, Accuracy: 93.52%
Epoch 5, Loss: 0.1244, Accuracy: 96.03%
Epoch 6, Loss: 0.0939, Accuracy: 97.65%
Epoch 7, Loss: 0.1009, Accuracy: 97.81%
Epoch 8, Loss: 0.0651, Accuracy: 98.46%
Epoch 9, Loss: 0.0728, Accuracy: 98.30%
Epoch 10, Loss: 0.0611, Accuracy: 98.22%


In [42]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

val_acc = 100 * correct / total
print(f"Validation Accuracy: {val_acc:.2f}%")


Validation Accuracy: 84.38%


In [44]:
import cv2
from torchvision import transforms

def predict_age(img_path):
    model.eval()
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (32, 32))  # same size used in CNN
    img = transforms.ToTensor()(img).unsqueeze(0).to(device)  # convert to tensor and add batch dim
    
    with torch.no_grad():
        output = model(img)
        _, pred = output.max(1)
        return class_names[pred.item()]

# Example usage
print(predict_age("../Data/Images/Training/Teen Naruto/Teent Naruto 003.jpg"))


Teen Naruto


In [46]:
# Save only weights
torch.save(model.state_dict(), "naruto_cnn_weights.pth")
